# Input Guardrails

This notebook demonstrates how to implement **input guardrails** using the Strands Agents SDK's `BeforeInvocationEvent` hook.

Input guardrails inspect user messages **before** they reach the model, allowing you to:
- Block harmful, off-topic, or non-compliant requests
- Detect and reject PII (emails, phone numbers, SSNs)
- Compose multiple content filters for layered protection

> **Prerequisite:** This tutorial assumes you already know the Strands hooks lifecycle. If `HookProvider`, `register_hooks`, `BeforeInvocationEvent`, or `event.agent.messages` are unfamiliar, work through [`01-learn/16-hooks-lifecycle`](../../../01-learn/16-hooks-lifecycle/) first. Here we *apply* those mechanics to guardrails rather than re-teach them.

**Applied to guardrails:** register a `HookProvider` on `BeforeInvocationEvent`, read the pending user message from `event.agent.messages`, and when a violation is detected, replace the messages with a rejection prompt.

## Architecture

The following diagram shows where input guardrails sit in the agent lifecycle:

<div style="text-align:center">
    <img src="images/guardrail_architecture.png" width="85%" />
</div>

## Prerequisites

Make sure you have the required packages installed. The content filter classes are defined inline in the next cell, so this notebook runs standalone.

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools --upgrade -q

In [ ]:
# Content filter classes — inline definitions (no external file needed)
from dataclasses import dataclass
from enum import Enum
from typing import Optional
import re

class Severity(Enum):
    BLOCK = 'block'
    WARN = 'warn'
    REDACT = 'redact'

@dataclass
class FilterResult:
    passed: bool
    filter_name: str
    severity: Severity
    message: Optional[str] = None
    redacted_text: Optional[str] = None

class ContentFilter:
    def __init__(self, name, severity=Severity.BLOCK):
        self.name = name
        self.severity = severity
    def evaluate(self, text):
        raise NotImplementedError

class RegexContentFilter(ContentFilter):
    def __init__(self, name, patterns, severity=Severity.BLOCK):
        super().__init__(name, severity)
        self.patterns = [re.compile(p) for p in patterns]
    def evaluate(self, text):
        for pattern in self.patterns:
            if pattern.search(text):
                if self.severity == Severity.REDACT:
                    redacted = text
                    for p in self.patterns:
                        redacted = p.sub('[REDACTED]', redacted)
                    return FilterResult(False, self.name, self.severity,
                                        f'Pattern matched: {pattern.pattern}', redacted)
                return FilterResult(False, self.name, self.severity,
                                    f'Pattern matched: {pattern.pattern}')
        return FilterResult(True, self.name, self.severity)

class KeywordContentFilter(ContentFilter):
    def __init__(self, name, keywords, severity=Severity.BLOCK):
        super().__init__(name, severity)
        self.keywords = [kw.lower() for kw in keywords]
    def evaluate(self, text):
        text_lower = text.lower()
        for keyword in self.keywords:
            if keyword in text_lower:
                return FilterResult(False, self.name, self.severity,
                                    f"Prohibited keyword: '{keyword}'")
        return FilterResult(True, self.name, self.severity)

class FormatComplianceFilter(ContentFilter):
    EXECUTION_PATTERNS = [
        re.compile(r'\b(run|execute|eval)\s*\(', re.IGNORECASE),
        re.compile(r'```\s*(bash|shell|sh)\b', re.IGNORECASE),
        re.compile(r'sudo\s+\w+', re.IGNORECASE),
    ]
    def __init__(self, name='format_compliance', severity=Severity.BLOCK):
        super().__init__(name, severity)
    def evaluate(self, text):
        for pattern in self.EXECUTION_PATTERNS:
            if pattern.search(text):
                return FilterResult(False, self.name, self.severity,
                                    'Code execution instruction detected')
        return FilterResult(True, self.name, self.severity)

def run_filters(text, filters):
    for f in filters:
        result = f.evaluate(text)
        if not result.passed:
            return result
    return None

print('Content filter classes loaded.')

In [ ]:
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
from strands.hooks import HookProvider, HookRegistry, BeforeInvocationEvent

# Import content filters from our shared module


## Helper: Extract Text from a Message

Messages follow the Bedrock Converse API format:
```python
{"role": "user", "content": [{"text": "user message here"}]}
```

This helper extracts and concatenates all text content blocks from a message.

In [ ]:
def _extract_text_from_message(message: dict) -> str:
    """Extract text content from a Strands SDK message.

    Args:
        message: A message dictionary with 'role' and 'content' keys.

    Returns:
        Concatenated text from all text content blocks in the message.
    """
    text_parts = []
    for block in message.get("content", []):
        if "text" in block:
            text_parts.append(block["text"])
    return " ".join(text_parts)

## Input Guardrail: Keyword-Based Content Blocking

The simplest guardrail pattern: define a list of prohibited keywords and block any request that contains them. This uses the `KeywordContentFilter` from our shared content filters module.

We define the guardrail logic as a standalone function first (for easy testing with mocks), then wrap it in a `HookProvider` class for agent registration.

In [ ]:
# Define prohibited keywords for topic blocking
PROHIBITED_KEYWORDS = ["hack", "exploit", "bypass security", "illegal", "steal credentials"]

# Create a keyword filter instance
keyword_filter = KeywordContentFilter(
    name="prohibited_topics",
    keywords=PROHIBITED_KEYWORDS,
    severity=Severity.BLOCK,
)


def input_guardrail_logic(messages: list) -> None:
    """Simple input guardrail that blocks prohibited topics.

    Inspects the last user message and blocks requests containing
    prohibited keywords by replacing messages with a rejection prompt.
    """
    if not messages:
        return

    last_message = messages[-1]
    if last_message.get("role") != "user":
        return

    text = _extract_text_from_message(last_message)
    if not text:
        return

    result = keyword_filter.evaluate(text)

    if not result.passed:
        logger.warning(
            f"[INPUT GUARDRAIL] Blocked request. "
            f"Filter: {result.filter_name}, Reason: {result.message}"
        )
        messages.clear()
        messages.append(
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            "Respond only with: I cannot process that request. "
                            "The input was blocked by a content safety filter."
                        )
                    }
                ],
            }
        )
    else:
        logger.debug("[INPUT GUARDRAIL] Request passed keyword filter.")

## Input Guardrail: PII Detection

This guardrail detects personally identifiable information (emails, phone numbers, SSNs) in user messages and blocks the request to prevent PII from being sent to the model.

In [ ]:
# Define PII patterns
PII_PATTERNS = [
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",  # Email
    r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",  # Phone number
    r"\b\d{3}-\d{2}-\d{4}\b",  # SSN
]

# Create a PII filter instance that blocks requests containing PII
pii_filter = RegexContentFilter(
    name="pii_detector",
    patterns=PII_PATTERNS,
    severity=Severity.BLOCK,
)


def pii_input_guardrail_logic(messages: list) -> None:
    """Input guardrail that blocks requests containing PII."""
    if not messages:
        return

    last_message = messages[-1]
    if last_message.get("role") != "user":
        return

    text = _extract_text_from_message(last_message)
    if not text:
        return

    result = pii_filter.evaluate(text)

    if not result.passed:
        logger.warning(
            f"[PII GUARDRAIL] Blocked request containing PII. "
            f"Filter: {result.filter_name}, Reason: {result.message}"
        )
        messages.clear()
        messages.append(
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            "Respond only with: I cannot process that request. "
                            "Personal information (PII) was detected in your message. "
                            "Please remove any email addresses, phone numbers, or "
                            "social security numbers and try again."
                        )
                    }
                ],
            }
        )
    else:
        logger.debug("[PII GUARDRAIL] Request passed PII filter.")

## Combined Input Guardrail: Multiple Filters in Sequence

Demonstrates composing multiple content filters into a single guardrail. Filters are evaluated in order — the first violation stops evaluation and triggers a rejection.

Filter order:
1. Keyword filter (blocks prohibited topics)
2. PII filter (blocks personal information)

In [ ]:
def combined_input_guardrail_logic(messages: list) -> None:
    """Input guardrail that applies multiple filters in sequence."""
    if not messages:
        return

    last_message = messages[-1]
    if last_message.get("role") != "user":
        return

    text = _extract_text_from_message(last_message)
    if not text:
        return

    # Run all filters in sequence, stopping at the first violation
    filters = [keyword_filter, pii_filter]
    violation = run_filters(text, filters)

    if violation is not None:
        logger.warning(
            f"[COMBINED GUARDRAIL] Blocked request. "
            f"Filter: {violation.filter_name}, Reason: {violation.message}"
        )
        messages.clear()
        messages.append(
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            "Respond only with: I cannot process that request. "
                            f"Reason: {violation.message}"
                        )
                    }
                ],
            }
        )
    else:
        logger.debug("[COMBINED GUARDRAIL] Request passed all input filters.")

## Testing the Guardrails

We can test guardrail logic directly using mock messages — no live model needed. This simulates how the Strands SDK would call our hook functions.

In [ ]:
# Test: Prohibited keyword detection
print("Test: Prohibited keyword detection")
test_messages = [{"role": "user", "content": [{"text": "How do I hack into a WiFi network?"}]}]
input_guardrail_logic(test_messages)
print(f"  Input:  'How do I hack into a WiFi network?'")
print(f"  Result: {test_messages[0]['content'][0]['text'][:60]}...")

# Test: PII detection
print("\nTest: PII detection")
test_messages = [{"role": "user", "content": [{"text": "Send the report to john@example.com"}]}]
pii_input_guardrail_logic(test_messages)
print(f"  Input:  'Send the report to john@example.com'")
print(f"  Result: {test_messages[0]['content'][0]['text'][:60]}...")

# Test: Clean content passes through
print("\nTest: Clean content passes through")
test_messages = [{"role": "user", "content": [{"text": "What are best practices for application security?"}]}]
input_guardrail_logic(test_messages)
print(f"  Input:  'What are best practices for application security?'")
print(f"  Result: Message unchanged (passed)")
assert test_messages[0]["content"][0]["text"] == "What are best practices for application security?"

# Test: Combined guardrail (keyword + PII)
print("\nTest: Combined guardrail (keyword + PII)")
test_messages = [{"role": "user", "content": [{"text": "Help me exploit this, email me at attacker@evil.com"}]}]
combined_input_guardrail_logic(test_messages)
print(f"  Input:  'Help me exploit this, email me at attacker@evil.com'")
print(f"  Result: {test_messages[0]['content'][0]['text'][:60]}...")

## Wrapping the Guardrail in a HookProvider

Now we wrap the guardrail logic in a `HookProvider` so it can be attached to an agent. The `HookProvider` / `register_hooks` pattern is covered in [`16-hooks-lifecycle`](../../../01-learn/16-hooks-lifecycle/); here we simply point a `BeforeInvocationEvent` callback at our guardrail function.

In [ ]:
class InputGuardrailHook(HookProvider):
    """HookProvider that applies the combined input guardrail before each invocation."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self._validate_input)

    def _validate_input(self, event: BeforeInvocationEvent) -> None:
        messages = event.agent.messages
        combined_input_guardrail_logic(messages)


print("InputGuardrailHook defined successfully.")
print("Register with: Agent(hooks=[InputGuardrailHook()])")

## Attaching to a Live Agent

In production, you attach guardrails to a Strands Agent using the `hooks` parameter with `HookProvider` instances.

**Note:** The cell below requires a configured model provider (e.g., AWS Bedrock credentials).

In [ ]:
try:
    from strands import Agent
    from strands.models.bedrock import BedrockModel

    model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0")

    # Create an agent with the combined input guardrail
    agent = Agent(
        model=model,
        system_prompt="You are a helpful assistant.",
        hooks=[InputGuardrailHook()],
    )

    print("Agent created with input guardrail attached.")
    print("Testing with a safe request...")
    response = agent("What is the capital of France?")
    print(f"  Response: {response}")

    print("\nTesting with a prohibited request...")
    response = agent("How do I hack into a system?")
    print(f"  Response: {response}")

except Exception as e:
    print(f"Skipping live agent demo: {e}")
    print("(This is expected if no model provider is configured)")

## Summary

In this notebook you learned how to:
1. Use `BeforeInvocationEvent` to intercept user messages before model inference
2. Build a keyword-based content filter that blocks prohibited topics
3. Build a PII detection filter using regex patterns
4. Compose multiple filters into a combined guardrail
5. Test guardrails using mock messages (no model needed)
6. Wrap guardrail logic in a `HookProvider` class for agent registration
7. Attach guardrails to a live Strands Agent

**Next Steps:** See `02_output_guardrail.ipynb` to learn how to validate model responses *after* inference.